# NISAR L1 RSLC (Level-1 Range-Doppler Single Look Complex) Product Tutorial
Author: Heresh Fattahi, July 2026

This tutorial provides a guide to working with NISAR Level-1 L-band RSLC products, including accessing, reading, and analyzing the data.

## Table of Contents
1. [Introduction to RSLC Products](#introduction)
2. [Understanding Granule Naming Convention](#granule-naming)
3. [Downloading Data from ASF DAAC](#downloading)
4. [Exploring the HDF5 Structure](#hdf5-structure)
5. [Working with RSLC Swaths](#swaths)
6. [Reading and Visualizing RSLC Data](#visualization)
7. [Understanding Swath Metadata](#swath-metadata)
8. [Valid Samples and PRF](#valid-samples)
9. [Geolocation Grid and Interpolation](#geolocation)
10. [Noise Equivalent Backscatter](#noise)
11. [Chirp Weighting Functions](#chirp-weighting)
12. [Orbit Information](#orbit)
13. [Radar Geometry Transformations with ISCE3](#isce3-transforms)

## 1. Introduction to RSLC Products <a name="introduction"></a>

The NISAR L1 RSLC (Range Single Look Complex) product contains focused synthetic aperture radar (SAR) data in slant range geometry. Key characteristics:

- **Complex-valued data**: Contains both amplitude and phase information
- **Native slant range geometry**: Data is in the sensor's acquisition geometry
- **Dual-frequency capable**: L-band (frequency A) and S-band (frequency B)
- **Multi-polarimetric**: Supports HH, HV, VH, VV polarizations
- **HDF5 format**: Hierarchical data structure with imagery and extensive metadata

RSLC products are essential for:
- Interferometric SAR (InSAR) processing
- Polarimetric analysis
- Advanced SAR applications requiring phase information

In [ ]:
# Import required libraries
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.interpolate import RegularGridInterpolator
import os

# Optional: ISCE3 for orbit and geometry transformations
try:
    from isce3.core import DateTime, Orbit, LUT2d
    from isce3.geometry import rdr2geo, geo2rdr
    from isce3.core import Ellipsoid
    ISCE3_AVAILABLE = True
except ImportError:
    print("ISCE3 not available. Some functionality will be limited.")
    ISCE3_AVAILABLE = False

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Understanding Granule Naming Convention <a name="granule-naming"></a>

Let's decode the example granule name:

```
NISAR_L1_PR_RSLC_026_101_A_026_2005_DHDH_A_20260727T044437_20260727T044512_P05023_N_F_J_001
```

Breaking down each component:

| Component | Value | Description |
|-----------|-------|-------------|
| Mission | NISAR | NASA-ISRO SAR mission |
| Level | L1 | Processing level (Level-1) |
| Processing Type | PR |PR = Production, UR=Urgent Response, OD=On-Demand |
| Product | RSLC | Range Single Look Complex |
| Cycle Number | 009 | Mission cycle number |
| Track number | 172 | Relative orbit number within cycle |
| Orbit Direction | A | Ascending (A) or Descending (D) |
| Frame Number | 008 | Track number |
| Bandwidth mode | 2005 | 4 chars for Bandwidth Mode Code of Primary and Secondary Bands |
| Polarization | DHDH |  4 chars for Polarization of the data for the primary and secondary bands. SH = HH – Single Polarity (H transmit and receive): SV = VV – Single Polarity (V transmit and receive), DH = HH/HV – Dual Polarity (H transmit), DV = VV/VH – Dual Polarity (V transmit), CL= LH/LV – Compact Polarity (Left transmit), CR = RH/RV – Compact Polarity (Right transmit), QP = HH/HV/VV/VH – Quad Polarity, NA if band does not exist |
| Source of Data | A | Acquired source of observation (A for single mode), Mixed mode of observations (M)|
| Start Time | 20260109T024620 | Radar start time in YYYYMMDDTHHMMSS (UTC) format|
| End Time | 20260109T024654 | Radar stop time in YYYYMMDDTHHMMSS (UTC) format |
| Composite Release Identifier (CRID) | P05023 | Version of the Data system producing the product |
| Orbit/pointing accuracy | N | 1 char for Product Accuracy or Fidelity of the Orbit Ephermis and Radar Pointing. P: Precise, M: Medium accuracy, N: Near Real Time accuracy, F: Forecast |
| Product Coverage | F | 1 char as Coverage Indicator: F for Full or P for Partial|
| Processor location | J | 1 char to represent the location of the Science Data System. J for JPL/NASA, N: for NRSC/ISRO. |
| Counter | 001 | Granule counter |

In [ ]:
# Example granule information
rslc_file = "NISAR_L1_PR_RSLC_026_101_A_026_2005_DHDH_A_20260727T044437_20260727T044512_P05023_N_F_J_001.h5"
granule_id = os.path.basename(rslc_file).replace(".h5", "") 
def parse_granule_name(granule_id):
    """Parse NISAR RSLC granule name into components"""
    parts = granule_id.split('_')
    return {
        'mission': parts[0],
        'level': parts[1],
        'product_type': parts[2],
        'product': parts[3],
        'cycle': int(parts[4]),
        'track': int(parts[5]),
        'orbit_direction': parts[6],
        'frane': int(parts[7]),
        'range_bandwidth': int(parts[8]),
        'freq_pol': parts[9],
        'source_of_data': parts[10],
        'start_time': parts[11],
        'end_time': parts[12],
        'crid': parts[13],
        'orbit_attitude_fidelity': parts[14],
        'coverage': parts[15],
        'processing_center': parts[16],
        'product_counter': parts[17]
    }

granule_info = parse_granule_name(granule_id)
print("Granule Information:")
for key, value in granule_info.items():
    print(f"  {key:20s}: {value}")

## 3. Downloading Data from ASF DAAC <a name="downloading"></a>

NISAR RSLC products are available from the Alaska Satellite Facility (ASF) DAAC.

### Option 1: Manual Download from ASF Vertex
1. Visit https://search.asf.alaska.edu/
2. Search for "NISAR" and filter by product type "RSLC"
3. Download the granule

### Option 2: Programmatic Download (requires authentication)
```python
# Example using asf_search library (requires installation)
# pip install asf_search

### For running this notebook you may get a granule with wget. Copy and paste the following command to a terminal:
wget https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_PROVISIONAL_V1/NISAR_L1_PR_RSLC_026_101_A_026_2005_DHDH_A_20260727T044437_20260727T044512_P05023_N_F_J_001/NISAR_L1_PR_RSLC_026_101_A_026_2005_DHDH_A_20260727T044437_20260727T044512_P05023_N_F_J_001.h5
#For this tutorial, we'll assume the file is already downloaded or accessible.

In [ ]:
# Specify the path to your RSLC file
# Update this path to point to your downloaded RSLC file
rslc_file = f"{granule_id}.h5"

# Check if file exists
if not os.path.exists(rslc_file):
    print(f"File not found: {rslc_file}")
    print("Please update the path to your RSLC file. If you don't have an RSLC file, please download one. See the cell above")
else:
    print(f"File found: {rslc_file}")
    file_size = os.path.getsize(rslc_file) / (1024**3)  # Size in GB
    print(f"File size: {file_size:.2f} GB")

## 4. Exploring the HDF5 Structure <a name="hdf5-structure"></a>

NISAR RSLC products use HDF5 format with a hierarchical structure. Let's explore the file organization.

In [ ]:
def print_hdf5_structure(name, obj, indent=0):
    """Recursively print HDF5 structure"""
    spacing = '  ' * indent
    if isinstance(obj, h5py.Group):
        print(f"{spacing}{name}/ (Group)")
    elif isinstance(obj, h5py.Dataset):
        print(f"{spacing}{name} (Dataset): shape={obj.shape}, dtype={obj.dtype}")

# Open the file and explore top-level structure
with h5py.File(rslc_file, 'r') as f:
    print("\n=== Top-level HDF5 Structure ===")
    print_hdf5_structure('/', f)
    
    for key in f.keys():
        print_hdf5_structure(key, f[key], indent=1)
        if key == 'science':
            for key2 in f[key].keys():
                print_hdf5_structure(key2, f[key][key2], indent=2)
                if key2 == 'LSAR':
                    for key3 in f[key][key2].keys():
                        print_hdf5_structure(key3, f[key][key2][key3], indent=3)

In [ ]:
def explore_hdf5_structure(filename, max_depth=5):
    """
    Recursively explore and print HDF5 file structure
    """
    def print_structure(name, obj, depth=0):
        if depth > max_depth:
            return
        
        indent = '  ' * depth
        if isinstance(obj, h5py.Group):
            print(f"{indent}📁 {name}/")
        elif isinstance(obj, h5py.Dataset):
            print(f"{indent}📄 {name} {obj.shape} {obj.dtype}")
    
    with h5py.File(filename, 'r') as f:
        print(f"\nHDF5 File: {filename}\n")
        print("=" * 80)
        f.visititems(print_structure)

# Example usage (uncomment when you have a GUNW file):
explore_hdf5_structure(rslc_file, max_depth=6)

### Exploring the Identification Group

The identification group contains metadata about the product.

In [ ]:
with h5py.File(rslc_file, 'r') as f:
    print("\n=== Identification Group ===")
    id_group = f['/science/LSAR/identification']
    
    # Print all attributes in identification group
    for key in id_group.keys():
        try:
            value = id_group[key][()]
            if isinstance(value, bytes):
                value = value.decode('utf-8')
            print(f"  {key:30s}: {value}")
        except:
            print(f"  {key:30s}: <Unable to read>")

In [ ]:
#import geopandas as gpd
from shapely.wkt import loads

metadata_path = "science/LSAR/identification/boundingPolygon"

# 2. Extract the WKT string from the HDF5 file
with h5py.File(rslc_file, "r") as f:
    wkt_bytes = f[metadata_path][()]
    wkt_string = wkt_bytes.decode("utf-8") if isinstance(wkt_bytes, bytes) else wkt_bytes

# 3. Parse WKT to a Shapely geometry
polygon_geom = loads(wkt_string)

# 4. Extract X (Longitude) and Y (Latitude) coordinates
# .exterior.xy works for Polygon geometries
x, y = polygon_geom.exterior.xy

# 5. Plot directly using standard matplotlib
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x, y, color='blue', linewidth=2)
ax.fill(x, y, color='blue', alpha=0.1) # Optional: fills the polygon safely

ax.set_title("NISAR RSLC Bounding Polygon")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.grid(True)
plt.show()

## 5. Working with RSLC Swaths <a name="swaths"></a>

NISAR RSLC data is organized into **swaths** under `/science/LSAR/RSLC/swaths/`. Depending on the mode of acquisition this group may contain multiple SLC images at different frequencies (frequency A or frequency B) acquired at different polarizations. The following figure visualizes all possible NISAR LSAR acquisition modes. The total possible range bandwidth of 77 MHz may be acquired in different modes such as those with 20 or 40 MHz on the main band (stroed in frequency A) and 5 MHz on the right side of the band (stored in frequency B). When the radar acquires data with the entire 77 MHz bandwidth, the RSLC product contains only frequency A group. 

<img src="./nisar_modes.png"> 



In [ ]:
with h5py.File(rslc_file, 'r') as f:
    list_of_frequencies = f['/science/LSAR/identification/listOfFrequencies'][()]
    swath_path = '/science/LSAR/RSLC/swaths'
    
    print("\n=== Available Frequencies ===")
    for freq in list_of_frequencies:
        freq = freq.decode('utf-8')
        print(f"\n{freq}:")
        freq_path = f"{swath_path}/frequency{freq}"
        
        # List available polarizations
        if 'listOfPolarizations' in f[freq_path]:
            pols = f[freq_path]['listOfPolarizations'][()]
            if isinstance(pols, bytes):
                pols = pols.decode('utf-8')
            print(f"  Polarizations: {pols}")
        
        # Find datasets (polarization channels)
        print("  Available datasets:")
        for item in f[freq_path].keys():
            if isinstance(f[freq_path][item], h5py.Dataset) and 'SubSwath' not in item:
                dataset = f[freq_path][item]
                print(f"    {item}: shape={dataset.shape}, dtype={dataset.dtype}")

Note the frequency A data has the same number of lines of the frequency B data. However, the number of samples in frequency B is a fraction of the number of pixels in frequency A proportinal to their range bandwidth ratio, i.e., 40 MHz RSLC is 8 times the 5 MHz RSLC and 20 MHz data is 4 times the 5 MHz data in range direction. 

## 6. Reading and Visualizing RSLC Data <a name="visualization"></a>

RSLC files can be very large (tens of GB). We'll use:
- **Striding**: Read every Nth pixel to reduce memory
- **Subsetting**: Read only a portion of the image

In [ ]:
def read_rslc_data(h5_file, frequency='frequencyA', polarization='HH', 
                   subswath=1, subset=None, stride=(1, 1)):
    """
    Read RSLC data with subsetting and striding options
    
    Parameters:
    -----------
    h5_file : str or h5py.File
        Path to RSLC file or open file handle
    frequency : str
        'frequencyA' or 'frequencyB'
    polarization : str
        Polarization channel (e.g., 'HH', 'HV', 'VV', 'VH')
    subswath : int
        Sub-swath number (usually 1)
    subset : tuple or None
        ((row_start, row_end), (col_start, col_end)) for spatial subset
    stride : tuple
        (row_stride, col_stride) for downsampling
    
    Returns:
    --------
    data : complex numpy array
        SLC data
    """
    should_close = False
    if isinstance(h5_file, str):
        f = h5py.File(h5_file, 'r')
        should_close = True
    else:
        f = h5_file
    
    try:
        # Construct dataset path
        dataset_path = f'/science/LSAR/RSLC/swaths/{frequency}/{polarization}'
        
        if dataset_path not in f:
            raise ValueError(f"Dataset not found: {dataset_path}")
        
        dataset = f[dataset_path]
        
        # Determine slice indices
        if subset is None:
            row_slice = slice(None, None, stride[0])
            col_slice = slice(None, None, stride[1])
        else:
            row_slice = slice(subset[0][0], subset[0][1], stride[0])
            col_slice = slice(subset[1][0], subset[1][1], stride[1])
        
        # Read data
        data = dataset[row_slice, col_slice]
        
        return data
    
    finally:
        if should_close:
            f.close()

# Read a subset with stride
# Adjust these parameters based on your file size and memory

window_size_rows = 50000
window_size_cols = 50000
pol = "HH"
stride = (10,1)

with h5py.File(rslc_file, 'r') as f:
    # Get full dimensions
    dataset_path = '/science/LSAR/RSLC/swaths/frequencyA/HH'
    full_shape = f[dataset_path].shape
    print(f"Full image dimensions: {full_shape}")
    
    # Read center subset with stride for quick look
    row_center = full_shape[0] // 2
    col_center = full_shape[1] // 2
    
    
    subset = (
        (max(0, row_center - window_size_rows), min(full_shape[0], row_center + window_size_rows)),
        (max(0, col_center - window_size_cols), min(full_shape[1], col_center + window_size_cols))
    )
    
    slc_data_freq_A = read_rslc_data(f, frequency='frequencyA', polarization=pol,
                              subset=subset, stride=stride)

    dataset_path = '/science/LSAR/RSLC/swaths/frequencyB/HH'
    full_shape = f[dataset_path].shape
    print(f"Full image dimensions: {full_shape}")
    
    # Read center subset with stride for quick look
    row_center = full_shape[0] // 2
    col_center = full_shape[1] // 2
    
    window_size_cols = window_size_cols//4
    
    subset = (
        (max(0, row_center - window_size_rows), min(full_shape[0], row_center + window_size_rows)),
        (max(0, col_center - window_size_cols), min(full_shape[1], col_center + window_size_cols))
    )
    slc_data_freq_B = read_rslc_data(f, frequency='frequencyB', polarization=pol,
                              subset=subset, stride=stride)
    
    print(f"Loaded subset shape (frequency A): {slc_data_freq_A.shape}")
    print(f"Loaded subset shape (frequency B): {slc_data_freq_B.shape}")
    print(f"Stride used for loading: f{stride}")

In [ ]:
# Visualize the SLC data
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Amplitude of freq A in dB
amplitude_db_A = 20 * np.log10(np.abs(slc_data_freq_A))
im1 = axes[0].imshow(amplitude_db_A, cmap='gray', aspect='auto', vmin=np.percentile(amplitude_db_A, 5),
                     vmax=np.percentile(amplitude_db_A, 95))
axes[0].set_title('Frequency A, Amplitude (dB)')
axes[0].set_xlabel('Range')
axes[0].set_ylabel('Azimuth')
plt.colorbar(im1, ax=axes[0], label='dB')

# Amplitude of freq B in dB
amplitude_db_B = 20 * np.log10(np.abs(slc_data_freq_B))
im2 = axes[1].imshow(amplitude_db_B, cmap='gray', aspect='auto', vmin=np.percentile(amplitude_db_B, 5),
                     vmax=np.percentile(amplitude_db_B, 95))
axes[1].set_title('Frequency B, Amplitude (dB)')
axes[1].set_xlabel('Range')
axes[1].set_ylabel('Azimuth')
plt.colorbar(im2, ax=axes[1], label='dB')

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Data Statistics ===")
print(f"Mean amplitude: {np.mean(amplitude_db_A):.2f}")
print(f"Std amplitude: {np.std(amplitude_db_A):.2f}")
print(f"Min amplitude: {np.min(amplitude_db_A):.2f}")
print(f"Max amplitude: {np.max(amplitude_db_A):.2f}")

## 7. Understanding Swath Metadata <a name="swath-metadata"></a>

Each frequency group contains extensive metadata about the SAR acquisition and processing parameters.

In [ ]:
with h5py.File(rslc_file, 'r') as f:
    freq_path = '/science/LSAR/RSLC/swaths/frequencyA'
    swath_path = '/science/LSAR/RSLC/swaths'
    print("\n=== Frequency A Metadata ===")
    
    # Key scalar parameters
    params_to_read = [
        'acquiredCenterFrequency',
        'acquiredRangeBandwidth',
        'acquiredAzimuthBandwidth',
        'nominalAcquisitionPRF',
        'centerFrequency',
        'processedRangeBandwidth',
        'processedAzimuthBandwidth',
        'slantRange',
        'slantRangeSpacing',
        'sceneCenterAlongTrackSpacing',
        'zeroDopplerTime',
        'zeroDopplerTimeSpacing'
    ]
    
    for param in params_to_read:
        #print(param)
        if param in f[freq_path]:
            value = f[freq_path][param][()]
            if isinstance(value, bytes):
                value = value.decode('utf-8')
            #if isinstance(value, np.ndarray):
            #    if value.size == 1:
            #        value = value.item()
            #    else:
            #        value = f"{value.shape} array (min={np.min(value):.3f}, max={np.max(value):.3f})"
            print(f"  {param:40s}: {value}")
        elif param in f[swath_path]:
            value = f[swath_path][param][()]
            print(f"  {param:40s}: {value}")

In [ ]:
speed_of_light = 299792458.0 # m/s
with h5py.File(rslc_file, 'r') as f:
    freq_path = '/science/LSAR/RSLC/swaths/frequencyA'
    range_bandwidth_A = f[freq_path]["processedRangeBandwidth"][()]
    range_spacing_A = f[freq_path]["slantRangeSpacing"][()]

    # range sampling frequency for frequency A data
    fs_A = speed_of_light/range_spacing_A/2

    freq_path = '/science/LSAR/RSLC/swaths/frequencyB'
    range_bandwidth_B = f[freq_path]["processedRangeBandwidth"][()]
    range_spacing_B = f[freq_path]["slantRangeSpacing"][()]

    # range sampling frequency for frequency B data
    fs_B = speed_of_light/range_spacing_B/2

print(f"Slant range bandwidth for frequency A: {range_bandwidth_A/1e6} MHz")
print(f"Slant range bandwidth for frequency B: {range_bandwidth_B/1e6} MHz")

print(f"range sampling frequency for frequency A: {fs_A/1e6} MHz")
print(f"range sampling frequency for frequency B: {fs_B/1e6} MHz")

Note that the sampling frequency is 1.2x of the range bandwidth.

The unit of slant ranges is meters and the azimuth time vector is provided as relative seconds since a refernce epoch.

In [ ]:
with h5py.File(rslc_file, 'r') as f:
    freq_path = '/science/LSAR/RSLC/swaths/frequencyA'
    swath_path = '/science/LSAR/RSLC/swaths'
    ds = f[f"{freq_path}/slantRange"]
    for attr_name in ds.attrs:
        attr_value = ds.attrs[attr_name]
        print(attr_name, ":", attr_value.decode("utf-8"))

    ds = f[f"{swath_path}/zeroDopplerTime"]
    for attr_name in ds.attrs:
        attr_value = ds.attrs[attr_name]
        print(attr_name, ":", attr_value.decode("utf-8"))

# Plotting Range Spectrum

In [ ]:
def compute_range_spectrum(slc_data, fs, line_fraction=1.0):
    """
    Computes the average range spectrum of an RSLC product.
    
    Parameters:
    -----------
    slc_data : numpy.ndarray
        2D complex array of SLC data (rows: slow-time/azimuth, cols: fast-time/range).
    fs : float
        Range sampling frequency in Hz.
    line_fraction : float, optional
        Fraction of azimuth lines (0.0 to 1.0) to use for the FFT average. Default is 1.0.
        
    Returns:
    --------
    frequencies : numpy.ndarray
        1D array of frequency bins in MHz.
    mean_spectrum_db : numpy.ndarray
        1D array of the averaged range spectrum in decibels (dB).
    """
    # 1. Determine number of azimuth lines to use
    num_azimuth_lines = slc_data.shape[0]
    num_lines_to_use = int(np.ceil(num_azimuth_lines * line_fraction))
    num_lines_to_use = max(1, min(num_lines_to_use, num_azimuth_lines))
    
    # Subsample the data along the azimuth direction
    subsampled_data = slc_data[:num_lines_to_use, :]
    
    # 2. Compute FFT along the range dimension (axis=1)
    # Apply a Hanning window along range to suppress sidelobes if desired,
    # but a raw FFT shows the true processed bandwidth filter edges.
    range_fft = np.fft.fft(subsampled_data, axis=1)
    range_fft_shifted = np.fft.fftshift(range_fft, axes=1)
    
    # 3. Calculate magnitude squared (power spectrum)
    power_spectrum = np.abs(range_fft_shifted) ** 2
    
    # 4. Average across the selected azimuth lines
    mean_power_spectrum = np.mean(power_spectrum, axis=0)
    
    # 5. Convert to decibels (dB) and normalize to peak
    mean_spectrum_db = 10 * np.log10(mean_power_spectrum + 1e-12)
    #mean_spectrum_db -= np.max(mean_spectrum_db)
    
    # 6. Generate frequency axis in MHz
    num_range_pixels = slc_data.shape[1]
    frequencies = np.fft.fftshift(np.fft.fftfreq(num_range_pixels, d=1/fs))
    frequencies_mhz = frequencies / 1e6
    
    return frequencies_mhz, mean_spectrum_db




In [ ]:
#Compute the spectrum using 50% of the available range lines
freqs_A, spectrum_A = compute_range_spectrum(slc_data_freq_A, fs=fs_A, line_fraction=0.5)
freqs_B, spectrum_B = compute_range_spectrum(slc_data_freq_B, fs=fs_B, line_fraction=0.5)

plt.figure(figsize=(10, 5), dpi=100)
plt.plot(freqs_A, spectrum_A, color='#1f77b4', linewidth=1.5, label='Averaged Range Spectrum')

# Formatting the plot
plt.title('NISAR RSLC Averaged Range Spectrum (frequency A)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Frequency (MHz)', fontsize=12)
plt.ylabel('Relative Power (dB)', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.ylim([-5, 55])  # Standard dynamic range for visualization
plt.xlim([freqs_A.min(), freqs_A.max()])

# Visual anchors for clarity
plt.axhline(0, color='black', linewidth=0.8, linestyle='-')
plt.legend(loc='lower center', frameon=True, shadow=False)
plt.tight_layout()


plt.figure(figsize=(10, 5), dpi=100)
plt.plot(freqs_B, spectrum_B, color='#1f77b4', linewidth=1.5, label='Averaged Range Spectrum')

# Formatting the plot
plt.title('NISAR RSLC Averaged Range Spectrum (frequency B)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Frequency (MHz)', fontsize=12)
plt.ylabel('Relative Power (dB)', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.ylim([-5, 55])  # Standard dynamic range for visualization
plt.xlim([freqs_B.min(), freqs_B.max()])

# Visual anchors for clarity
plt.axhline(0, color='black', linewidth=0.8, linestyle='-')
plt.legend(loc='lower center', frameon=True, shadow=False)
plt.tight_layout()


# Plotting Azimuth Spectrum

In [ ]:
with h5py.File(rslc_file, 'r') as f:
    swath_path = '/science/LSAR/RSLC/swaths/'
    azimuth_time_spacing = f[swath_path]["zeroDopplerTimeSpacing"][()]
    prf = 1.0/azimuth_time_spacing

print(f"Pulse Repetition Frequency (PRF): {prf}")

In [ ]:
def compute_azimuth_spectrum(slc_data, prf, range_fraction=1.0):
    """
    Computes the average azimuth (Doppler) spectrum of an RSLC product.
    
    Parameters:
    -----------
    slc_data : numpy.ndarray
        2D complex array of SLC data (rows: slow-time/azimuth, cols: fast-time/range).
    prf : float
        Pulse Repetition Frequency in Hz.
    range_fraction : float, optional
        Fraction of range pixels (0.0 to 1.0) to use for the FFT average. Default is 1.0.
        
    Returns:
    --------
    frequencies : numpy.ndarray
        1D array of frequency bins (Doppler frequencies) in Hz.
    mean_spectrum_db : numpy.ndarray
        1D array of the averaged azimuth spectrum in decibels (dB).
    """
    # 1. Determine number of range lines (columns) to use
    num_range_pixels = slc_data.shape[1]
    num_cols_to_use = int(np.ceil(num_range_pixels * range_fraction))
    num_cols_to_use = max(1, min(num_cols_to_use, num_range_pixels))
    
    # Subsample the data along the range direction
    subsampled_data = slc_data[:, :num_cols_to_use]
    
    # 2. Compute FFT along the azimuth dimension (axis=0)
    azimuth_fft = np.fft.fft(subsampled_data, axis=0)
    azimuth_fft_shifted = np.fft.fftshift(azimuth_fft, axes=0)
    
    # 3. Calculate magnitude squared (power spectrum)
    power_spectrum = np.abs(azimuth_fft_shifted) ** 2
    
    # 4. Average across the selected range pixels
    mean_power_spectrum = np.mean(power_spectrum, axis=1)
    
    # 5. Convert to decibels (dB) and normalize to peak
    mean_spectrum_db = 10 * np.log10(mean_power_spectrum + 1e-12)
    mean_spectrum_db -= np.max(mean_spectrum_db)
    
    # 6. Generate frequency axis in Hz (centered around 0 Hz)
    num_azimuth_lines = slc_data.shape[0]
    frequencies_hz = np.fft.fftshift(np.fft.fftfreq(num_azimuth_lines, d=1/prf))
    
    return frequencies_hz, mean_spectrum_db




In [ ]:
window_size_rows = 55000
window_size_cols = 5000
pol = "HH"
stride = (1,1)

with h5py.File(rslc_file, 'r') as f:
    # Get full dimensions
    dataset_path = '/science/LSAR/RSLC/swaths/frequencyA/HH'
    full_shape = f[dataset_path].shape
    print(f"Full image dimensions: {full_shape}")
    
    # Read center subset with stride for quick look
    row_center = full_shape[0] // 2
    col_center = full_shape[1] // 2
    
    
    subset = (
        (max(0, row_center - window_size_rows), min(full_shape[0], row_center + window_size_rows)),
        (max(0, col_center - window_size_cols), min(full_shape[1], col_center + window_size_cols))
    )
    
    slc_data_freq_A = read_rslc_data(f, frequency='frequencyA', polarization=pol,
                              subset=subset, stride=stride)

    dataset_path = '/science/LSAR/RSLC/swaths/frequencyB/HH'
    full_shape = f[dataset_path].shape
    print(f"Full image dimensions: {full_shape}")
    
    # Read center subset with stride for quick look
    row_center = full_shape[0] // 2
    col_center = full_shape[1] // 2
    
    window_size_cols = window_size_cols//4
    
    subset = (
        (max(0, row_center - window_size_rows), min(full_shape[0], row_center + window_size_rows)),
        (max(0, col_center - window_size_cols), min(full_shape[1], col_center + window_size_cols))
    )
    slc_data_freq_B = read_rslc_data(f, frequency='frequencyB', polarization=pol,
                              subset=subset, stride=stride)
    
    print(f"Loaded subset shape (frequency A): {slc_data_freq_A.shape}")
    print(f"Loaded subset shape (frequency B): {slc_data_freq_B.shape}")
    print(f"Stride used for loading: f{stride}")

In [ ]:


freqs_az, spectrum_az_A = compute_azimuth_spectrum(slc_data_freq_A, prf=prf, range_fraction=0.4)
freqs_az, spectrum_az_B = compute_azimuth_spectrum(slc_data_freq_B, prf=prf, range_fraction=0.4)

# 3. Generate a clean, publication-ready plot
plt.figure(figsize=(10, 5), dpi=100)
plt.plot(freqs_az, spectrum_az_A, color='#2ca02c', linewidth=1.5, label='Averaged Azimuth Spectrum')

# Formatting the plot
plt.title('NISAR RSLC Averaged Azimuth (Doppler) Spectrum', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Doppler Frequency (Hz)', fontsize=12)
plt.ylabel('Relative Power (dB)', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.ylim([-35, 5])  # Standard dynamic range for visualization
plt.xlim([freqs_az.min(), freqs_az.max()])

# Visual anchors for clarity
plt.axhline(0, color='black', linewidth=0.8, linestyle='-')
plt.axvline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.7, label='Zero Doppler')
plt.legend(loc='lower center', frameon=True, shadow=False)
plt.tight_layout()



## 8. Valid Samples and PRF <a name="valid-samples"></a>

The `validSamplesSubSwath*` datasets indicate which samples in each azimuth line contain valid data. This is important for:
- Variable PRF systems
- Identifying data gaps
- Processing efficiency

In [ ]:
with h5py.File(rslc_file, 'r') as f:
    # Check for different frequencies
    for freq in ['frequencyA', 'frequencyB']:
        freq_path = f'/science/LSAR/RSLC/swaths/{freq}'
        if freq_path not in f:
            continue
            
        print(f"\n=== {freq} Valid Samples ===")
        
        # Check for valid samples datasets
        subswath = 1
        valid_samples_path = f'{freq_path}/validSamplesSubSwath{subswath}'
        
        if valid_samples_path in f:
            valid_samples = f[valid_samples_path][:]
            print(f"Valid samples array shape: {valid_samples.shape}")
            print(f"Valid samples dimensions: {valid_samples.shape[0]} azimuth lines x {valid_samples.shape[1]} entries")
            
            # Valid samples format: [firstValidSample, lastValidSample] for each line
            first_valid = valid_samples[:, 0]
            last_valid = valid_samples[:, 1]
            num_valid = last_valid - first_valid + 1
            
            print(f"\nFirst valid sample range: {np.min(first_valid)} - {np.max(first_valid)}")
            print(f"Last valid sample range: {np.min(last_valid)} - {np.max(last_valid)}")
            print(f"Valid samples per line: min={np.min(num_valid)}, max={np.max(num_valid)}, mean={np.mean(num_valid):.1f}")
            

## 9. Geolocation Grid and Interpolation <a name="geolocation"></a>

The geolocation grid provides geographic coordinates and geometry parameters on a coarse grid. These can be interpolated to obtain values for any pixel.

The metadata cubes are in `/science/LSAR/RSLC/metadata/geolocationGrid/`:


In [ ]:
with h5py.File(rslc_file, 'r') as f:
    geo_grid_path = '/science/LSAR/RSLC/metadata/geolocationGrid'
    
    print("\n=== Geolocation Grid Structure ===")
    for item in f[geo_grid_path].keys():
        dataset = f[geo_grid_path][item]
        print(f"  {item:30s}: shape={dataset.shape}, dtype={dataset.dtype}")
    
    # Load grid coordinates
    azimuth_time = f[f'{geo_grid_path}/zeroDopplerTime'][:]
    slant_range = f[f'{geo_grid_path}/slantRange'][:]
    
    print(f"\nGeolocation grid dimensions:")
    print(f"  Azimuth time samples: {len(azimuth_time)}")
    print(f"  Slant range samples: {len(slant_range)}")
    print(f"  Azimuth time range: {azimuth_time[0]:.3f} - {azimuth_time[-1]:.3f} seconds")
    print(f"  Slant range: {slant_range[0]:.2f} - {slant_range[-1]:.2f} meters")

### Interpolating Geolocation Parameters

Let's create an interpolator for incidence angle and demonstrate how to query it at arbitrary positions.

In [ ]:
def create_geolocation_interpolator(h5_file, parameter='incidenceAngle'):
    """
    Create interpolator for geolocation grid parameters
    
    Parameters:
    -----------
    h5_file : h5py.File
        Open HDF5 file handle
    parameter : str
        Parameter name (e.g., 'incidenceAngle', 'losUnitVectorX', losUnitVectorY, etc)
    
    Returns:
    --------
    interpolator : RegularGridInterpolator
        Scipy interpolator object
    azimuth_time : array
        Azimuth time coordinates
    slant_range : array
        Slant range coordinates
    """
    geo_grid_path = '/science/LSAR/RSLC/metadata/geolocationGrid'
    
    azimuth_time = h5_file[f'{geo_grid_path}/zeroDopplerTime'][:]
    slant_range = h5_file[f'{geo_grid_path}/slantRange'][:]
    height = h5_file[f'{geo_grid_path}/heightAboveEllipsoid'][:]
    
    values = h5_file[f'{geo_grid_path}/{parameter}'][:]
    interpolator = RegularGridInterpolator(
        (height, azimuth_time, slant_range),
        values,
        method='linear',
        bounds_error=False,
        fill_value=np.nan
    )
    
    return interpolator, azimuth_time, slant_range

# Example: Create incidence angle interpolator
with h5py.File(rslc_file, 'r') as f:
    inc_interp, az_time, sr = create_geolocation_interpolator(f, 'incidenceAngle')
    
    # Query at specific points
    # Example: middle of the scene
    mid_az = (az_time[0] + az_time[-1]) / 2
    mid_sr = (sr[0] + sr[-1]) / 2
    
    query_points = np.array([[1000,mid_az, mid_sr]])
    inc_angle = inc_interp(query_points)[0]
    
    print(f"\nInterpolated incidence angle at scene center:")
    print(f"  Azimuth time: {mid_az:.3f} s")
    print(f"  Slant range: {mid_sr:.2f} m")
    print(f"Incidence angle: {inc_angle:.2f} degrees ")

## 12. Orbit Information <a name="orbit"></a>

The orbit metadata contains the satellite state vectors (position and velocity) used for SAR processing and geometric correction.

In [ ]:
with h5py.File(rslc_file, 'r') as f:
    orbit_path = '/science/LSAR/RSLC/metadata/orbit'
    
    print("\n=== Orbit Information ===")
    
    # Read orbit data
    orbit_time = f[f'{orbit_path}/time'][:]  # Seconds since reference epoch
    orbit_position = f[f'{orbit_path}/position'][:]  # ECEF coordinates [m]
    orbit_velocity = f[f'{orbit_path}/velocity'][:]  # ECEF velocity [m/s]

    print(f[f'{orbit_path}/time'].attrs['units'].decode('utf-8'))
    
    print(f"\nNumber of orbit state vectors: {len(orbit_time)}")
    print(f"Time span: {orbit_time[0]:.2f} - {orbit_time[-1]:.2f} seconds")
    print(f"Time spacing: ~{np.mean(np.diff(orbit_time)):.2f} seconds")
    
    print(f"\nPosition range (ECEF):")
    print(f"  X: {np.min(orbit_position[:, 0])/1000:.2f} - {np.max(orbit_position[:, 0])/1000:.2f} km")
    print(f"  Y: {np.min(orbit_position[:, 1])/1000:.2f} - {np.max(orbit_position[:, 1])/1000:.2f} km")
    print(f"  Z: {np.min(orbit_position[:, 2])/1000:.2f} - {np.max(orbit_position[:, 2])/1000:.2f} km")
    
    # Calculate orbit altitude (approximate)
    orbit_radius = np.linalg.norm(orbit_position, axis=1)
    earth_radius = 6371000  # meters
    altitude = orbit_radius - earth_radius
    print(f"\nOrbit altitude: {np.mean(altitude)/1000:.2f} km (mean)")
    print(f"Velocity magnitude: {np.mean(np.linalg.norm(orbit_velocity, axis=1)):.2f} m/s")

In [ ]:
# Visualize orbit
with h5py.File(rslc_file, 'r') as f:
    orbit_time = f['/science/LSAR/RSLC/metadata/orbit/time'][:]
    orbit_position = f['/science/LSAR/RSLC/metadata/orbit/position'][:]
    orbit_velocity = f['/science/LSAR/RSLC/metadata/orbit/velocity'][:]

fig = plt.figure(figsize=(15, 5))

# 3D orbit trajectory
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(orbit_position[:, 0]/1000, orbit_position[:, 1]/1000, orbit_position[:, 2]/1000)
ax1.set_xlabel('X (km)')
ax1.set_ylabel('Y (km)')
ax1.set_zlabel('Z (km)')
ax1.set_title('Orbit Trajectory (ECEF)')

# Position components vs time
ax2 = fig.add_subplot(132)
ax2.plot(orbit_time, orbit_position[:, 0]/1000, label='X')
ax2.plot(orbit_time, orbit_position[:, 1]/1000, label='Y')
ax2.plot(orbit_time, orbit_position[:, 2]/1000, label='Z')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Position (km)')
ax2.set_title('Position Components')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Velocity components vs time
ax3 = fig.add_subplot(133)
ax3.plot(orbit_time, orbit_velocity[:, 0], label='Vx')
ax3.plot(orbit_time, orbit_velocity[:, 1], label='Vy')
ax3.plot(orbit_time, orbit_velocity[:, 2], label='Vz')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Velocity (m/s)')
ax3.set_title('Velocity Components')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()